# 7 · Word-level recognition  (IPA & orthographic)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/07_word_level/07_word_level_recognition.ipynb)

An **alternative recognition target** to the phoneme model in Notebook 5.
Where stage 5 outputs a phoneme sequence, this notebook trains **word-level**
recognisers whose unit of evaluation is the whole word. It reuses the same
data-prep stages (1–4) and the same backbone (Wav2Vec2 XLS-R-300M + CTC), but
switches the tokenizer to a **character-level** `Wav2Vec2CTCTokenizer` in which
the space between words becomes the word delimiter `|`.

**Two types — pick one with the `TARGET` switch below:**

| | Target label | Vocab | Decoded output | Metric |
|---|---|---|---|---|
| **A · IPA word-level** | word-form IPA, e.g. `dat hœʁt` | IPA characters | `dat hœʁt` | WER / CER (+ derived PER) |
| **B · Orthographic** | Kölsch spelling, e.g. `un dann hammer` | letters (a–z, ä ö ü ß …) | `un dann hammer` | WER / CER |

**How this differs from phoneme recognition (Notebook 5)**
- Notebook 5 uses `Wav2Vec2PhonemeCTCTokenizer` with atomic phoneme tokens
  (`aː`, `t͡s` are single units) and reports a phoneme-token error rate.
- Here we use `Wav2Vec2CTCTokenizer`: each **character** is a token, multi-char
  IPA units split into code points, and the model learns word boundaries
  **natively** — the decoded output already has spaces, with no post-processing.
- A single wrong character fails the whole word, so **word-level WER is
  naturally higher than the phoneme error rate** — that is expected, not a bug.

## Setup

In [ ]:
!pip -q install "transformers>=4.40" datasets evaluate jiwer torchaudio accelerate
import torch, json, re, numpy as np, pandas as pd
from dataclasses import dataclass
from typing import Union
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "·", device)

In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

# On Google Colab: clone the repo once (or mount Drive and point _root at it).
try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/kolsch-tandem.git /content/kolsch-tandem")
except Exception:
    pass

# Repo root = the folder that contains kolsch_paths.py (found from any subfolder).
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, PAGES, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)                       # any remaining relative paths resolve at the root
print("repo root:", ROOT)

## 1 · Choose the target and load the manifest

Set **`TARGET`** to `"ipa"` or `"orthography"`. Each row of the manifest needs an
`audio_path`, a `split`, and the label column for your chosen target:
- `TARGET="ipa"` → an `ipa_wordform` column (word-form IPA, e.g. `dat hœʁt`)
  — this is the *un-tokenised* IPA from Notebook 4 (words, not phoneme-split).
- `TARGET="orthography"` → a `text` column (raw Kölsch spelling).

In [ ]:
TARGET = "ipa"            # "ipa"  or  "orthography"
LABEL_COL = {"ipa": "ipa_wordform", "orthography": "text"}[TARGET]

import pandas as pd, os, hashlib
man = pd.read_csv(os.path.join(SEG,"manifest.csv")).merge(
        pd.read_csv(INDEX)[["id","speaker"]], on="id", how="left")
def bucket(sp):
    h=int(hashlib.md5(str(sp).encode()).hexdigest(),16)%10
    return "test" if h<1 else "valid" if h<2 else "train"
man["split"]=man["speaker"].map(bucket)
df = man
print("TARGET =",TARGET,"| label column =",LABEL_COL,"| rows:",len(df))

## 2 · (Orthography only) normalise the spelling

Lower-case, keep Kölsch letters and apostrophes, drop other punctuation, and
**remove digits** (spell numerals out beforehand if they matter — the `\w`
class would otherwise keep `1998` as characters and pollute the vocab).
For `TARGET="ipa"` this is a no-op: the IPA word-form is used as-is.

In [ ]:
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"\d+", "", s)                       # drop digits
    s = re.sub(r"[^a-zäöüßçæœø' \t]", " ", s)       # keep Kölsch letters + apostrophe
    s = re.sub(r"\s+", " ", s).strip()
    return s

if TARGET == "orthography":
    df[LABEL_COL] = df[LABEL_COL].map(normalize_text)
df["label"] = df[LABEL_COL]                          # unified label column
print("sample labels:", df["label"].head(3).tolist())

## 3 · Character inventory + distribution plot

Same analysis style as the phoneme notebook, but the unit is the **character**.
Space is handled separately as the word delimiter, so it is excluded here.

In [ ]:
from collections import Counter
def char_counts(series):
    c = Counter()
    for s in series.astype(str):
        c.update(s.replace(" ", ""))
    return c

if len(df):
    splits = {k: char_counts(df[df.split==k]["label"]) for k in ["train","valid","test"]}
    chars = sorted(set().union(*[set(c) for c in splits.values()]))
    tot = {k: max(1,sum(c.values())) for k,c in splits.items()}
    x = np.arange(len(chars)); bottom = np.zeros(len(chars))
    fig, ax = plt.subplots(figsize=(16,6))
    for name,color in [("train","#1f77b4"),("valid","#ff7f0e"),("test","#2ca02c")]:
        vals = np.array([splits[name].get(ch,0)/tot[name] for ch in chars])
        ax.bar(x, vals, bottom=bottom, label=name, color=color); bottom += vals
    ax.set_xticks(x); ax.set_xticklabels(chars, rotation=45)
    ax.set_xlabel("character"); ax.set_ylabel("Ratio (stacked)")
    ax.set_title(f"{TARGET.title()} character distribution — {len(chars)} characters")
    ax.legend(); fig.tight_layout(); plt.savefig(f"{TARGET}_char_distribution.png", dpi=150); plt.show()
else:
    print("Add your manifest to see the character distribution.")

## 4 · Build the character vocabulary (space → `|`)

In [ ]:
def extract_chars(series):
    chars = set()
    for s in series.astype(str):
        chars.update(s.replace(" ", ""))            # space handled as '|' separately
    return chars

def build_char_vocab(train_labels, path="vocab.json"):
    vocab = {c: i for i, c in enumerate(sorted(extract_chars(train_labels)))}
    vocab["|"]     = len(vocab)                      # word delimiter (was space)
    vocab["[UNK]"] = len(vocab)
    vocab["[PAD]"] = len(vocab)
    json.dump(vocab, open(path,"w"), ensure_ascii=False)
    return vocab

# vocab = build_char_vocab(df[df.split=="train"]["label"])
# print(f"{TARGET} vocab size:", len(vocab))

## 5 · Character-level tokenizer + processor

In [ ]:
from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor,
                          Wav2Vec2Processor)

# Character-level: each character is a token; the space in the text maps to the
# word_delimiter '|' and decodes back to a space -> the model learns word
# boundaries natively (output is already "dat hœʁt" / "un dann hammer").
# tokenizer = Wav2Vec2CTCTokenizer("vocab.json", unk_token="[UNK]",
#                 pad_token="[PAD]", word_delimiter_token="|")
# fe = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0,
#                 do_normalize=True, return_attention_mask=True)
# processor = Wav2Vec2Processor(feature_extractor=fe, tokenizer=tokenizer)
print("character-level tokenizer ready (space -> | word delimiter)")

## 6 · Datasets, collator, metrics

In [ ]:
from datasets import Dataset, Audio

def to_ds(split):
    sub = df[df.split==split][["audio_path","label"]]
    return Dataset.from_pandas(sub, preserve_index=False).cast_column(
        "audio_path", Audio(sampling_rate=16000))

def prepare(batch):
    a = batch["audio_path"]
    batch["input_values"] = processor(a["array"], sampling_rate=16000).input_values[0]
    batch["labels"] = processor(text=batch["label"]).input_ids   # space -> |
    return batch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool,str] = True
    def __call__(self, features):
        inp=[{"input_values":f["input_values"]} for f in features]
        lab=[{"input_ids":f["labels"]} for f in features]
        batch=self.processor.pad(inp,padding=self.padding,return_tensors="pt")
        with self.processor.as_target_processor():
            lb=self.processor.pad(lab,padding=self.padding,return_tensors="pt")
        batch["labels"]=lb["input_ids"].masked_fill(lb.attention_mask.ne(1),-100)
        return batch

In [ ]:
import evaluate
wer_metric, cer_metric = evaluate.load("wer"), evaluate.load("cer")

def compute_metrics(pred):
    ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids==-100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(ids)                       # "dat hœʁt"
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    out = {"wer": wer_metric.compute(predictions=pred_str, references=label_str),
           "cer": cer_metric.compute(predictions=pred_str, references=label_str)}
    return out
# For TARGET="ipa" you can additionally derive a PER by re-tokenising the decoded
# IPA words into phonemes (reuse tokenize_ipa from Notebook 4) and aligning.

## 7 · Model + training (same backbone as Notebook 5)

In [ ]:
from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer

# model = Wav2Vec2ForCTC.from_pretrained(
#     "facebook/wav2vec2-xls-r-300m",
#     ctc_loss_reduction="mean", ctc_zero_infinity=True,
#     pad_token_id=processor.tokenizer.pad_token_id, vocab_size=len(processor.tokenizer))
# model.freeze_feature_encoder(); model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=os.path.join(MODELS, f"kolsch_wordlevel_{TARGET}"),
    group_by_length=True,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    per_device_eval_batch_size=2, num_train_epochs=150, fp16=True,
    learning_rate=3e-5, lr_scheduler_type="cosine", warmup_steps=500,
    weight_decay=0.05,
    eval_strategy="steps", eval_steps=1000, save_strategy="steps", save_steps=1000,
    logging_steps=1000, load_best_model_at_end=True,
    metric_for_best_model="wer", greater_is_better=False, save_total_limit=2)

# trainer = Trainer(model=model, args=args, data_collator=DataCollatorCTCWithPadding(processor),
#     train_dataset=to_ds("train").map(prepare, remove_columns=["audio_path","label"]),
#     eval_dataset=to_ds("valid").map(prepare, remove_columns=["audio_path","label"]),
#     compute_metrics=compute_metrics, tokenizer=processor.feature_extractor)
# trainer.train(); trainer.save_model(args.output_dir); processor.save_pretrained(args.output_dir)
print(f"training args ready for word-level ({TARGET}) — best checkpoint on validation WER")

## 8 · What to expect

- **IPA word-level** produces a phonetic transcription with word boundaries.
  Its WER is higher than the phoneme error rate from Notebook 5, because one
  wrong character fails the whole word — that is arithmetic, not a regression.
- **Orthographic** produces standard Kölsch spelling; its WER/CER depend heavily
  on how consistently the source spelling was normalised.
- Run this notebook twice (`TARGET="ipa"`, then `"orthography"`) to get both
  models, and compare against the phoneme recogniser (Notebook 5) using the
  same test split. Fill your numbers into the table in the repo README.